# Covariate-shift conformal filtering

Compare ordinary filtering with weighted label sets and weighted recall/precision calibration on the same external predictions. Calibration uses internal annotations and external features only.

## Load the data

Place the downloaded files in `data/`, or set `GFLD_DATA_DIR` to their location. Dataset details are provided in the paper.

In [ ]:
%matplotlib inline
from pathlib import Path
from dataclasses import asdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from gfld import *

# Works when Jupyter starts in the repository root or notebooks directory.
ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").is_file() else Path.cwd().parent
DATA_DIR = example_directory(ROOT)  # GFLD_DATA_DIR or repository data/
data = load_example(DATA_DIR)
print({k: v["images"] for k, v in data["metadata"]["splits"].items()})

In [ ]:
# Split calibration and evaluation by image.
_, calibration_idx, evaluation_idx = next(calibration_splits(
    len(data["internal"]), folds=5, permutations=1, seed=42))
calibration = [data["internal"][i] for i in calibration_idx]
evaluation = [data["internal"][i] for i in evaluation_idx]
annotations = data["internal_annotations"]
assert set(p.image_id for p in calibration).isdisjoint(p.image_id for p in evaluation)
print(f"{len(calibration)} calibration images; {len(evaluation)} held-out images")

In [ ]:
cp = calibrate_cp(calibration, annotations, alpha=0.01)
pd.DataFrame([asdict(cp)])

In [ ]:
filtering = calibrate_filtering(calibration, annotations, cp,
    target_recall=0.80, target_precision=0.85, delta=0.5)
pd.DataFrame([asdict(filtering.recall), asdict(filtering.precision)],
             index=["recall", "precision"])

In [ ]:
weighted_cp, cp_weights, domain_model = calibrate_shift_cp(
    calibration, annotations, data["internal_features"], data["external_features"],
    alpha=0.01, weight_cap=10.0)
pd.DataFrame([asdict(cp), asdict(weighted_cp)], index=["ordinary", "weighted"])

## Calibrate weighted filtering

Weight missed-object and false-discovery risks by image-level density estimates, then select the recall and precision thresholds.

In [ ]:
weighted_filtering = calibrate_shift_filtering(
    calibration, annotations, weighted_cp,
    data["internal_features"], data["external_features"],
    target_recall=0.80, target_precision=0.85, delta=0.5, weight_cap=10.0,
    relaxation_step=0.03, max_retries=10)
pd.DataFrame([asdict(weighted_filtering.recall), asdict(weighted_filtering.precision)],
             index=["weighted recall", "weighted precision"])

## Inspect calibration results

Check the selected thresholds, target feasibility, and any relaxation. A positive relaxation records a lower target used for calibration.

In [ ]:
for name, result in [("recall", weighted_filtering.recall), ("precision", weighted_filtering.precision)]:
    print(name, "original target:", result.target, "feasible:", result.feasible,
          "relaxation:", result.relaxation, "fallback:", result.fallback)

## Evaluate on external images

Compare the achieved recall, precision, and numbers of detections in each region.

In [ ]:
results = {}
for name, model in [("ordinary", filtering), ("weighted", weighted_filtering)]:
    results[name] = evaluate_filtering(data["external"], data["external_annotations"], model)
comparison = pd.DataFrame({name: {
    "candidate_recall": result["candidate"]["recall"],
    "confident_precision": result["confident"]["precision"],
    "recall_feasible_on_calibration": result["recall_feasible"],
    "precision_feasible_on_calibration": result["precision_feasible"],
    **result["regions"],
} for name, result in results.items()}).T
display(comparison)
pd.DataFrame({name: result["regions"] for name, result in results.items()}).plot.bar()
plt.ylabel("External detections"); plt.xticks(rotation=0); plt.tight_layout(); plt.show()